## Example 1: structured JSON logging with Logger

In [ ]:
import io
import json
import logging

import aws_lambda_powertools
from aws_lambda_powertools import Logger

log_buffer = io.StringIO()
logger = Logger(service="orders", logger_handler=logging.StreamHandler(log_buffer))
logger.append_keys(order_id="o-123")  # persists on every later record
logger.info("order created", extra={"amount": 42})
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("order failed")

info_record, error_record = (json.loads(line) for line in log_buffer.getvalue().splitlines())
assert info_record["level"] == "INFO" and info_record["message"] == "order created", info_record
assert info_record["service"] == "orders" and info_record["order_id"] == "o-123", info_record
assert info_record["amount"] == 42, info_record
assert error_record["level"] == "ERROR" and error_record["exception_name"] == "ZeroDivisionError", error_record
assert "division by zero" in error_record["exception"], error_record
print(f"✓ aws_lambda_powertools {aws_lambda_powertools.__version__}: Logger wrote "
      f"{sorted(info_record)} and an ERROR record with the traceback")

## Example 2: CloudWatch embedded metric format with Metrics

In [ ]:
from aws_lambda_powertools import Metrics
from aws_lambda_powertools.metrics import MetricUnit

# serialize_metric_set() builds the EMF document locally; nothing is sent to CloudWatch.
metrics = Metrics(namespace="ExampleApp", service="orders")
metrics.add_dimension(name="stage", value="test")
metrics.add_metric(name="OrdersCreated", unit=MetricUnit.Count, value=1)
metrics.add_metric(name="OrdersCreated", unit=MetricUnit.Count, value=2)
metrics.add_metric(name="Latency", unit=MetricUnit.Milliseconds, value=12.5)
document = metrics.serialize_metric_set()
metrics.clear_metrics()

directive = document["_aws"]["CloudWatchMetrics"][0]
assert directive["Namespace"] == "ExampleApp", directive
assert sorted(directive["Dimensions"][0]) == ["service", "stage"], directive["Dimensions"]
assert {m["Name"]: m["Unit"] for m in directive["Metrics"]} == {"OrdersCreated": "Count", "Latency": "Milliseconds"}
assert document["OrdersCreated"] == [1.0, 2.0] and document["Latency"] == [12.5], document
assert document["stage"] == "test" and document["service"] == "orders", document
assert isinstance(document["_aws"]["Timestamp"], int)
print(f"✓ EMF document: {json.dumps(document, sort_keys=True)[:110]}...")

## Example 3: routing API Gateway events with APIGatewayRestResolver

In [ ]:
from aws_lambda_powertools.event_handler import APIGatewayRestResolver

app = APIGatewayRestResolver()


@app.get("/orders/<order_id>")
def get_order(order_id: str):
    return {"order_id": order_id, "verbose": app.current_event.get_query_string_value("verbose")}


@app.post("/orders")
def create_order():
    return {"created": app.current_event.json_body["sku"]}, 201


def rest_event(method, path, body=None, query=None):
    """A minimal API Gateway REST (v1) proxy event, as Lambda would receive it."""
    return {
        "resource": path, "path": path, "httpMethod": method,
        "headers": {"Content-Type": "application/json"}, "multiValueHeaders": {},
        "queryStringParameters": query, "multiValueQueryStringParameters": None,
        "pathParameters": None, "stageVariables": None,
        "requestContext": {"stage": "test", "httpMethod": method, "path": path, "requestId": "request-1"},
        "body": body, "isBase64Encoded": False,
    }


got = app.resolve(rest_event("GET", "/orders/o-7", query={"verbose": "1"}), None)
assert got["statusCode"] == 200, got
assert json.loads(got["body"]) == {"order_id": "o-7", "verbose": "1"}, got["body"]

created = app.resolve(rest_event("POST", "/orders", body=json.dumps({"sku": "abc"})), None)
assert created["statusCode"] == 201 and json.loads(created["body"]) == {"created": "abc"}, created

missing = app.resolve(rest_event("DELETE", "/orders/o-7"), None)
assert missing["statusCode"] == 404, missing
print(f"✓ GET -> {got['statusCode']}, POST -> {created['statusCode']}, unrouted DELETE -> {missing['statusCode']}")

## Example 4: partial batch failures for SQS with BatchProcessor

In [ ]:
from aws_lambda_powertools.utilities.batch import BatchProcessor, EventType, process_partial_response

processed = []


def record_handler(record):
    payload = json.loads(record.body)
    if payload["qty"] < 0:
        raise ValueError(f"negative quantity in {record.message_id}")
    processed.append(payload["sku"])


def sqs_record(message_id, body):
    return {
        "messageId": message_id, "receiptHandle": f"handle-{message_id}", "body": json.dumps(body),
        "attributes": {"ApproximateReceiveCount": "1", "SentTimestamp": "1700000000000"},
        "messageAttributes": {}, "md5OfBody": "", "eventSource": "aws:sqs",
        "eventSourceARN": "arn:aws:sqs:us-east-1:123456789012:example-queue", "awsRegion": "us-east-1",
    }


event = {"Records": [
    sqs_record("m-1", {"sku": "a", "qty": 1}),
    sqs_record("m-2", {"sku": "b", "qty": -5}),
    sqs_record("m-3", {"sku": "c", "qty": 2}),
]}
response = process_partial_response(event=event, record_handler=record_handler,
                                    processor=BatchProcessor(event_type=EventType.SQS))
assert processed == ["a", "c"], processed
assert response == {"batchItemFailures": [{"itemIdentifier": "m-2"}]}, response
print(f"✓ processed {processed}; reported back to SQS for retry: {response['batchItemFailures']}")

## Example 5: typed event data classes

In [ ]:
from aws_lambda_powertools.utilities.data_classes import APIGatewayProxyEvent, S3Event

api_event = APIGatewayProxyEvent(
    rest_event("POST", "/orders", body=json.dumps({"sku": "abc"}), query={"dry_run": "true"})
)
assert api_event.http_method == "POST" and api_event.path == "/orders"
assert api_event.json_body == {"sku": "abc"}
assert api_event.get_query_string_value("dry_run") == "true"
assert api_event.headers.get("content-type") == "application/json", "header lookup is case-insensitive"
assert api_event.request_context.stage == "test"

# S3 sends object keys URL-encoded; the data class decodes them.
s3_event = S3Event({"Records": [{
    "eventVersion": "2.1", "eventSource": "aws:s3", "awsRegion": "us-east-1", "eventName": "ObjectCreated:Put",
    "eventTime": "2024-01-01T00:00:00.000Z",
    "s3": {"s3SchemaVersion": "1.0", "bucket": {"name": "example-bucket", "arn": "arn:aws:s3:::example-bucket"},
           "object": {"key": "reports/q1+summary%21.csv", "size": 1024, "eTag": "0123456789abcdef"}},
}]})
record = next(s3_event.records)
assert s3_event.bucket_name == "example-bucket"
assert record.event_name == "ObjectCreated:Put" and record.s3.get_object.size == 1024
assert s3_event.object_key == "reports/q1 summary!.csv", s3_event.object_key
print(f"✓ API Gateway event {api_event.http_method} {api_event.path}; S3 key decoded to {s3_event.object_key!r}")